In [ ]:
!pip -q install pytorchvideo av fvcore iopath

In [ ]:
from __future__ import annotations

import json
import random
import shutil
import zipfile
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from google.colab import drive
from pytorchvideo.models.hub import slowfast_r50

# Paths

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch')
DRIVE_MANIFEST_DIR = DRIVE_ROOT / 'manifests'
DRIVE_DOWNLOADS = DRIVE_ROOT / 'downloads'

LOCAL_RUNTIME_ROOT = Path('/content/charades_raw_from_drive')
MANIFEST_DIR = LOCAL_RUNTIME_ROOT / 'manifests'
VIDEO_ROOT = LOCAL_RUNTIME_ROOT / 'raw' / 'videos_480p' / 'Charades_v1_480'
RUN_ROOT = Path('/content/drive/MyDrive/momentlens_runs/charades_slowfast_phase1/run_001')
RUN_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_MANIFEST_PATH = MANIFEST_DIR / 'charades_raw_train_manifest.json'
VAL_MANIFEST_PATH = MANIFEST_DIR / 'charades_raw_val_manifest.json'
CLASS_MAP_PATH = MANIFEST_DIR / 'charades_class_map.json'

MANIFEST_FILES = [
    'charades_raw_manifest.json',
    'charades_raw_train_manifest.json',
    'charades_raw_val_manifest.json',
    'charades_raw_test_manifest.json',
    'charades_raw_survey.json',
    'charades_class_map.json',
    'charades_class_names.txt',
]

LOCAL_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_ROOT.parent.mkdir(parents=True, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def copy_file(src: Path, dst: Path):
    if not src.exists():
        raise FileNotFoundError(src)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return
    shutil.copy2(src, dst)


def rewrite_manifest_video_paths(manifest_path: Path):
    rows = json.loads(manifest_path.read_text())
    for row in rows:
        row['video_path'] = str((VIDEO_ROOT / f"{row['video_id']}.mp4").resolve())
    manifest_path.write_text(json.dumps(rows, indent=2, ensure_ascii=False))


for name in MANIFEST_FILES:
    copy_file(DRIVE_MANIFEST_DIR / name, MANIFEST_DIR / name)

for name in ['charades_raw_manifest.json', 'charades_raw_train_manifest.json', 'charades_raw_val_manifest.json', 'charades_raw_test_manifest.json']:
    rewrite_manifest_video_paths(MANIFEST_DIR / name)


In [ ]:
# Unzip videos directly from the zip stored on Drive into local runtime.
video_zip = DRIVE_DOWNLOADS / 'Charades_v1_480.zip'
# if not video_zip.exists():
#     raise FileNotFoundError(video_zip)
# with zipfile.ZipFile(video_zip, 'r') as zf:
#     zf.extractall(VIDEO_ROOT.parent)


In [ ]:
manifest_path = MANIFEST_DIR / 'charades_raw_survey.json'
train_manifest_path = MANIFEST_DIR / 'charades_raw_train_manifest.json'
val_manifest_path = MANIFEST_DIR / 'charades_raw_val_manifest.json'
class_map_path = MANIFEST_DIR / 'charades_class_map.json'

for path in [manifest_path, train_manifest_path, val_manifest_path, class_map_path]:
    if not path.exists():
        raise FileNotFoundError(path)

survey = json.loads(manifest_path.read_text())
train_rows = json.loads(train_manifest_path.read_text())
val_rows = json.loads(val_manifest_path.read_text())
class_map = json.loads(class_map_path.read_text())

print(f"dataset: train={len(train_rows)} val={len(val_rows)} classes={len(class_map['idx_to_name'])} avg_actions={survey['avg_actions_per_video']:.2f} avg_len={survey['avg_video_length_sec']:.2f}s")


dataset: train=7187 val=798 classes=157 avg_actions=6.75 avg_len=29.66s


In [ ]:
# Config

SEED = 42
CLIP_LEN_SEC = 6.0
POS_IOU = 0.25
NUM_FRAMES = 32
ALPHA = 4
BATCH_SIZE = 8
NUM_WORKERS = 12
EPOCHS = 5
START_EPOCH = 2
RESUME_FROM_BEST = True
BEST_MAP_INIT = 0.1161
LR = 3e-5
WEIGHT_DECAY = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

KINETICS_MEAN = torch.tensor([0.45, 0.45, 0.45]).view(1, 3, 1, 1)
KINETICS_STD = torch.tensor([0.225, 0.225, 0.225]).view(1, 3, 1, 1)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
with CLASS_MAP_PATH.open('r', encoding='utf-8') as f:
    class_map = json.load(f)

CLASS_TO_IDX = class_map['class_to_idx']
IDX_TO_NAME = class_map['idx_to_name']
NUM_CLASSES = len(IDX_TO_NAME)

with TRAIN_MANIFEST_PATH.open('r', encoding='utf-8') as f:
    train_rows = json.load(f)
with VAL_MANIFEST_PATH.open('r', encoding='utf-8') as f:
    val_rows = json.load(f)

for row in train_rows + val_rows:
    row['video_path'] = str((VIDEO_ROOT / f"{row['video_id']}.mp4").resolve())


for row in train_rows[:5] + val_rows[:5]:
    if not Path(row['video_path']).exists():
        raise FileNotFoundError(row['video_path'])


In [ ]:
def clip_iou(a_start, a_end, b_start, b_end):
    inter = max(0.0, min(a_end, b_end) - max(a_start, b_start))
    len_a = max(0.0, a_end - a_start)
    len_b = max(0.0, b_end - b_start)
    union = len_a + len_b - inter
    return inter / union if union > 0 else 0.0


def fit_clip(start_sec, video_len_sec):
    if video_len_sec <= CLIP_LEN_SEC:
        return 0.0, float(video_len_sec)
    start_sec = max(0.0, min(float(start_sec), float(video_len_sec - CLIP_LEN_SEC)))
    return start_sec, start_sec + CLIP_LEN_SEC


def clip_labels(clip_start, clip_end, intervals):
    labels = []
    for class_idx, (s, e) in intervals:
        if clip_iou(clip_start, clip_end, s, e) >= POS_IOU:
            labels.append(class_idx)
    return sorted(set(labels))


def choose_clip(row, is_train):
    intervals = list(zip(row['labels'], row['action_timings']))
    if not intervals:
        clip_start, clip_end = fit_clip(0.0, row['length'])
        return {
            'video_id': row['video_id'],
            'video_path': row['video_path'],
            'clip_start': round(float(clip_start), 2),
            'clip_end': round(float(clip_end), 2),
            'label_indices': [],
            'sample_type': 'bg',
        }

    class_idx, (s, e) = random.choice(intervals) if is_train else intervals[0]
    dur = e - s
    if dur <= CLIP_LEN_SEC:
        center = (s + e) / 2.0
        clip_start, clip_end = fit_clip(center - CLIP_LEN_SEC / 2.0, row['length'])
    else:
        start_min = s
        start_max = max(s, e - CLIP_LEN_SEC)
        start_sec = random.uniform(start_min, start_max) if is_train else start_min
        clip_start, clip_end = fit_clip(start_sec, row['length'])

    labels = clip_labels(clip_start, clip_end, intervals)
    if not labels:
        labels = [class_idx]
    return {
        'video_id': row['video_id'],
        'video_path': row['video_path'],
        'clip_start': round(float(clip_start), 2),
        'clip_end': round(float(clip_end), 2),
        'label_indices': labels,
        'sample_type': 'pos',
    }


def build_clip_bundle(item, is_train):
    if 'clip_start' in item and 'clip_end' in item and 'video_path' in item:
        sample = item
    else:
        sample = choose_clip(item, is_train)
    x = decode_clip_frames(sample['video_path'], sample['clip_start'], sample['clip_end'])
    slow, fast = pack_pathways(x)

    target = torch.zeros(NUM_CLASSES, dtype=torch.float32)
    if sample['label_indices']:
        target[sample['label_indices']] = 1.0
    return slow, fast, target


In [ ]:
VIDEO_DECODE_IMAGE_SIZE = 224

def decode_clip_frames(video_path, clip_start, clip_end, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'cannot open video: {video_path}')

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 24.0)
    start_frame = max(0, int(round(clip_start * fps)))
    end_frame = min(total_frames - 1, int(round(clip_end * fps)))
    if end_frame < start_frame:
        end_frame = start_frame

    target_count = min(num_frames, end_frame - start_frame + 1)
    frame_indices = np.linspace(start_frame, end_frame, num=max(1, target_count), dtype=np.int64)

    frames = []
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (VIDEO_DECODE_IMAGE_SIZE, VIDEO_DECODE_IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        frames.append(torch.from_numpy(frame).permute(2, 0, 1).contiguous().to(torch.uint8))

    cap.release()

    if not frames:
        raise RuntimeError(f'No frames decoded from: {video_path}')

    if len(frames) < num_frames:
        frames.extend([frames[-1].clone() for _ in range(num_frames - len(frames))])

    clip_frames = torch.stack(frames)
    idx = np.linspace(0, clip_frames.shape[0] - 1, num_frames).astype(np.int64)
    clip_frames = clip_frames[idx].to(torch.float32).div(255.0)
    x = (clip_frames - KINETICS_MEAN) / KINETICS_STD
    return x

def pack_pathways(frames):
    fast = frames.permute(1, 0, 2, 3).contiguous()
    slow = fast[:, ::ALPHA, :, :].contiguous()
    return [slow, fast]




In [ ]:
class CharadesSlowFastDataset(Dataset):
    def __init__(self, rows, num_classes, is_train):
        self.rows = rows
        self.num_classes = num_classes
        self.is_train = is_train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return build_clip_bundle(self.rows[idx], is_train=self.is_train)


train_ds = CharadesSlowFastDataset(train_rows, NUM_CLASSES, is_train=True)
val_ds = CharadesSlowFastDataset(val_rows, NUM_CLASSES, is_train=False)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4,
)


In [ ]:
model = slowfast_r50(pretrained=True)
head = model.blocks[-1]
if not hasattr(head, 'proj'):
    raise RuntimeError('Unexpected SlowFast head layout')
head.proj = nn.Linear(head.proj.in_features, NUM_CLASSES)
model = model.to(DEVICE)

pos_counts = np.zeros(NUM_CLASSES, dtype=np.float32)
for row in train_rows:
    for idx in row['labels']:
        pos_counts[idx] += 1.0

neg_counts = len(train_rows) - pos_counts
pos_weight = np.where(pos_counts > 0, neg_counts / np.maximum(pos_counts, 1.0), 1.0)
pos_weight = np.clip(pos_weight, 1.0, 20.0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler('cuda')


/tmp/ipykernel_12177/2802969458.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


In [ ]:
def mean_average_precision(y_true, y_score):
    aps = []
    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() == 0:
            continue
        aps.append(average_precision_score(y_true[:, c], y_score[:, c]))
    return float(np.mean(aps)) if aps else 0.0


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_targets = []
    for slow, fast, target in tqdm(loader, desc='val', leave=False):
        slow = slow.to(DEVICE, non_blocking=True)
        fast = fast.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        with autocast('cuda'):
            logits = model([slow, fast])
            loss = criterion(logits, target)
        total_loss += loss.item() * target.size(0)
        all_probs.append(torch.sigmoid(logits).detach().cpu())
        all_targets.append(target.detach().cpu())
    probs = torch.cat(all_probs, dim=0).numpy()
    targets = torch.cat(all_targets, dim=0).numpy()
    return {
        'loss': total_loss / len(loader.dataset),
        'mAP': mean_average_precision(targets, probs),
    }


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for slow, fast, target in tqdm(loader, desc='train', leave=False):
        slow = slow.to(DEVICE, non_blocking=True)
        fast = fast.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            logits = model([slow, fast])
            loss = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * target.size(0)
    return total_loss / len(loader.dataset)


In [ ]:
history = []
loaded_resume = RESUME_FROM_BEST and best_path.exists()
best_map = BEST_MAP_INIT
best_path = RUN_ROOT / 'best.pt'
if loaded_resume:
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print(f'loaded checkpoint: {best_path}')

effective_start_epoch = START_EPOCH if loaded_resume else 0
for epoch in range(effective_start_epoch, EPOCHS):
    train_loss = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader)
    scheduler.step()

    row = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_metrics['loss'],
        'val_mAP': val_metrics['mAP'],
        'lr': scheduler.get_last_lr()[0],
    }
    history.append(row)
    print(f"epoch {row['epoch']}: train={row['train_loss']:.4f} val={row['val_loss']:.4f} mAP={row['val_mAP']:.4f} lr={row['lr']:.2e}")

    if val_metrics['mAP'] > best_map:
        best_map = val_metrics['mAP']
        torch.save(model.state_dict(), best_path)

with (RUN_ROOT / 'history.json').open('w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)

import pandas as pd
pd.DataFrame(history).to_csv(RUN_ROOT / 'history.csv', index=False)
with (RUN_ROOT / 'config.json').open('w', encoding='utf-8') as f:
    json.dump({
        'seed': SEED,
        'clip_len_sec': CLIP_LEN_SEC,
        'pos_iou': POS_IOU,
        'num_frames': NUM_FRAMES,
        'alpha': ALPHA,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'start_epoch': START_EPOCH,
        'resume_from_best': RESUME_FROM_BEST,
        'best_map_init': BEST_MAP_INIT,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'num_classes': NUM_CLASSES,
        'train_rows': len(train_rows),
        'val_rows': len(val_rows),
    }, f, indent=2)

print(f'best mAP={best_map:.4f}')
print(f'best checkpoint={best_path}')


train:   0%|          | 0/1797 [00:00<?, ?it/s]

/tmp/ipykernel_12177/2133409703.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


val:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_12177/2133409703.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


epoch 1: train=0.5654 val=0.5150 mAP=0.0841 lr=2.71e-05


...

/tmp/ipykernel_12177/2133409703.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


epoch 2: train=0.4948 val=0.4813 mAP=0.1161 lr=1.96e-05


train:   0%|          | 0/1797 [00:00<?, ?it/s]

/tmp/ipykernel_12177/2133409703.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


KeyboardInterrupt: 